In [4]:
import os
import sys

# Silence Hugging Face Hub unauthenticated & telemetry warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"

sys.path.append("../")

import pandas as pd
from app.services.scoring_engine import compute_match_score
from app.services.resume_parser import get_resume_text

df = pd.read_csv("../data/Resume.csv")

In [5]:
job_description = "Backend engineer with Python, FastAPI, PostgreSQL, Docker experience."
required_skills = ["python", "fastapi", "postgresql", "docker"]

print(df["Category"].unique())

<StringArray>
[                    'HR',               'DESIGNER', 'INFORMATION-TECHNOLOGY',
                'TEACHER',               'ADVOCATE',   'BUSINESS-DEVELOPMENT',
             'HEALTHCARE',                'FITNESS',            'AGRICULTURE',
                    'BPO',                  'SALES',             'CONSULTANT',
          'DIGITAL-MEDIA',             'AUTOMOBILE',                   'CHEF',
                'FINANCE',                'APPAREL',            'ENGINEERING',
             'ACCOUNTANT',           'CONSTRUCTION',       'PUBLIC-RELATIONS',
                'BANKING',                   'ARTS',               'AVIATION']
Length: 24, dtype: str


In [6]:
target_category = "INFORMATION-TECHNOLOGY"  # set this to an exact value from Cell 2's output

matching = df[df["Category"] == target_category].sample(n=min(10, len(df[df["Category"] == target_category])), random_state=1)
non_matching = df[df["Category"] != target_category].sample(n=10, random_state=1)

def score_group(group_df):
    scores = []
    for _, row in group_df.iterrows():
        text = get_resume_text(raw_text=row["Resume_str"])
        result = compute_match_score(text, job_description, required_skills)
        scores.append(result["match_score"])
    return scores

matching_scores = score_group(matching)
non_matching_scores = score_group(non_matching)

print(f"Matching category avg score:     {sum(matching_scores)/len(matching_scores):.4f}")
print(f"Non-matching category avg score: {sum(non_matching_scores)/len(non_matching_scores):.4f}")

Matching category avg score:     0.3201
Non-matching category avg score: 0.2669


In [7]:
from scipy.stats import mannwhitneyu

stat, p_value = mannwhitneyu(matching_scores, non_matching_scores, alternative="greater")
print(f"p-value: {p_value:.4f}")
if p_value < 0.05:
    print("Matching category scores significantly higher — scoring engine is discriminating well.")
else:
    print("No significant difference — may need to tune weights/taxonomy.")

p-value: 0.0023
Matching category scores significantly higher — scoring engine is discriminating well.
